# Agents and Tools

Workflows shine when *you* know the steps. **Agents** shine when you don't: give Claude a **goal + a set of tools** and let it decide how to combine them. Build the agent once, and it handles varied, unpredictable requests — at the cost of some reliability/predictability (next lesson).

**The power is in combining simple tools.** With three datetime tools —
`get_current_datetime`, `add_duration_to_datetime`, `set_reminder` — the *same* agent handles:
- "What's the time?" → one tool
- "What day is it in 11 days?" → get current, then add duration
- "Remind me to hit the gym next Wednesday" → all three in sequence

Claude can even ask for missing info ("when did you buy it?" before computing a warranty expiry).

> Runnable illustration — it reuses the exact tools + agent loop from the *tool-use* section, now framed as an agent. On Haiku 4.5.

## The agent = tools + a loop (reused from the tool-use section)

In [1]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

import json
from datetime import datetime, timedelta
from anthropic import Anthropic
from anthropic.types import Message, ToolParam

client = Anthropic()
model = "claude-haiku-4-5-20251001"


def add_user_message(messages, m):
    messages.append({"role": "user", "content": m.content if isinstance(m, Message) else m})

def add_assistant_message(messages, m):
    messages.append({"role": "assistant", "content": m.content if isinstance(m, Message) else m})

def text_from_message(message):
    return "\n".join(b.text for b in message.content if b.type == "text")

def chat(messages, tools=None):
    params = {"model": model, "max_tokens": 1000, "messages": messages, "temperature": 1.0}
    if tools:
        params["tools"] = tools
    return client.messages.create(**params)


# --- tools ---
def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    return datetime.now().strftime(date_format)

def add_duration_to_datetime(datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"):
    date = datetime.strptime(datetime_str, input_format)
    if unit == "seconds": nd = date + timedelta(seconds=duration)
    elif unit == "minutes": nd = date + timedelta(minutes=duration)
    elif unit == "hours": nd = date + timedelta(hours=duration)
    elif unit == "days": nd = date + timedelta(days=duration)
    elif unit == "weeks": nd = date + timedelta(weeks=duration)
    elif unit == "years": nd = date.replace(year=date.year + duration)
    else: raise ValueError(f"Unsupported unit: {unit}")
    return nd.strftime("%A, %B %d, %Y %I:%M:%S %p")

def set_reminder(content, timestamp):
    print(f"----\n(REMINDER SET for {timestamp}: {content})\n----")
    return "Reminder set."

In [2]:
get_current_datetime_schema = ToolParam({
    "name": "get_current_datetime",
    "description": "Returns the current date and time formatted per the given strftime format string.",
    "input_schema": {"type": "object", "properties": {
        "date_format": {"type": "string", "description": "strftime format; default %Y-%m-%d %H:%M:%S", "default": "%Y-%m-%d %H:%M:%S"}}, "required": []},
})

add_duration_to_datetime_schema = ToolParam({
    "name": "add_duration_to_datetime",
    "description": "Adds a duration (in a given unit) to a datetime string and returns a detailed formatted datetime.",
    "input_schema": {"type": "object", "properties": {
        "datetime_str": {"type": "string", "description": "Input datetime, parsed per input_format"},
        "duration": {"type": "number", "description": "Amount to add (can be negative)"},
        "unit": {"type": "string", "description": "seconds|minutes|hours|days|weeks|years"},
        "input_format": {"type": "string", "description": "strptime format for datetime_str; default %Y-%m-%d"}},
        "required": ["datetime_str"]},
})

set_reminder_schema = ToolParam({
    "name": "set_reminder",
    "description": "Creates a reminder with the given content at the given ISO-8601 timestamp.",
    "input_schema": {"type": "object", "properties": {
        "content": {"type": "string", "description": "What to be reminded about"},
        "timestamp": {"type": "string", "description": "ISO-8601 time to trigger the reminder"}},
        "required": ["content", "timestamp"]},
})

TOOLS = [get_current_datetime_schema, add_duration_to_datetime_schema, set_reminder_schema]


def run_tool(name, tool_input):
    return {
        "get_current_datetime": get_current_datetime,
        "add_duration_to_datetime": add_duration_to_datetime,
        "set_reminder": set_reminder,
    }[name](**tool_input)


def run_tools(message):
    blocks = []
    for req in [b for b in message.content if b.type == "tool_use"]:
        try:
            out = run_tool(req.name, req.input)
            blocks.append({"type": "tool_result", "tool_use_id": req.id, "content": json.dumps(out), "is_error": False})
        except Exception as e:
            blocks.append({"type": "tool_result", "tool_use_id": req.id, "content": f"Error: {e}", "is_error": True})
    return blocks


def run_agent(user_text, verbose=True):
    messages = []
    add_user_message(messages, user_text)
    while True:
        response = chat(messages, tools=TOOLS)
        add_assistant_message(messages, response)
        if verbose:
            for b in response.content:
                if b.type == "tool_use":
                    print(f"  -> tool: {b.name}({b.input})")
        if response.stop_reason != "tool_use":
            return text_from_message(response)
        add_user_message(messages, run_tools(response))

## Same agent, different tool combinations

One agent, no per-request logic — Claude decides which tools to chain.

In [3]:
for q in [
    "What's the current time?",
    "What day of the week will it be 11 days from now?",
    "Set a reminder to go to the gym next Wednesday at 6pm.",
]:
    print(f"\nUSER: {q}")
    print("ANSWER:", run_agent(q))


USER: What's the current time?
  -> tool: get_current_datetime({})
ANSWER: The current time is **2026-07-13 21:15:02** (9:15 PM and 2 seconds on July 13th, 2026).

USER: What day of the week will it be 11 days from now?
  -> tool: get_current_datetime({'date_format': '%Y-%m-%d'})
  -> tool: add_duration_to_datetime({'datetime_str': '2026-07-13', 'duration': 11, 'unit': 'days'})
ANSWER: 11 days from now will be **Friday, July 24, 2026**.

USER: Set a reminder to go to the gym next Wednesday at 6pm.
  -> tool: get_current_datetime({})
  -> tool: add_duration_to_datetime({'datetime_str': '2026-07-13', 'duration': 10, 'unit': 'days'})
  -> tool: set_reminder({'content': 'Go to the gym', 'timestamp': '2026-07-22T18:00:00'})
----
(REMINDER SET for 2026-07-22T18:00:00: Go to the gym)
----
ANSWER: Perfect! I've set a reminder for you to go to the gym on Wednesday, July 22, 2026 at 6:00 PM.


## Tools should be abstract, not hyper-specialized

The best agents get **generic, combinable** tools. **Claude Code** is the exemplar — it has `bash`, `read`, `write`, `edit`, `glob`, `grep`, and *not* a `refactor_code` or `install_dependencies` tool. Claude composes the primitives to handle scenarios the developers never explicitly planned.

**Best practice — combinable tools.** A social-video agent might expose:
- `bash` (FFMPEG for video processing)
- `generate_image`
- `text_to_speech`
- `post_media`

That set supports both a straight "create and post a video" run *and* interactive flows (generate a sample image → get approval → proceed) — the agent adapts to feedback, which a rigid workflow can't.

## Takeaway

An agent is just **goal + tools + the tool-use loop** (the exact loop you built earlier) — the difference from a workflow is that **Claude owns the control flow**. Give it a few abstract, composable tools and it generalizes to tasks you didn't script.

The catch — flexibility trades against reliability, latency, and cost — is the subject of the next lesson.

Next: **Environment inspection**, then **Workflows vs agents**.